# Setup

In [ ]:
%pip install mistralai pandas numpy

In [ ]:
from mistralai.client import Mistral
import os
import json
import time

# Initialize the Mistral client
api_key = "YOUR_API_KEY_HERE"
API_KEY = os.environ.get("MISTRAL_API_KEY", api_key)
client = Mistral(api_key=API_KEY)

# RAG: Retrieval-Augmented Generation

In [ ]:
"""
RAG combines information retrieval with generation to provide
more accurate and contextual responses based on external knowledge.
"""

def basic_prompt(question, model="mistral-small-latest"):

    messages = [{"role": "user", "content": question}]

    response = client.chat.complete(model=model, messages=messages)

    return response.choices[0].message.content

# Sample knowledge base, here provide your own knowledge base
KNOWLEDGE_BASE = [
    {
        "id": 1,
        "content": "The Eiffel Tower is a wrought-iron lattice tower located in Paris, France. It was constructed from 1887 to 1889 as the entrance arch for the 1889 World's Fair."
    },
    {
        "id": 2,
        "content": "Python is a high-level, interpreted programming language created by Guido van Rossum and first released in 1991. It emphasizes code readability and uses significant indentation."
    },
    {
        "id": 3,
        "content": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing algorithms that can access data and use it to learn for themselves."
    },
    {
        "id": 4,
        "content": "The Great Wall of China is a series of fortifications built along the historical northern borders of ancient Chinese states. Construction began in the 7th century BC and continued through the 17th century AD."
    }
]

def retrieval_demo(query, knowledge_base, top_k=2):
    """
    Simple keyword-based retrieval (alternatively, try to use vector embeddings or other retrival methods)

    Args:
        query (str): User's question / prompt
        knowledge_base (list): List of documents
        top_k (int): Number of documents to retrieve

    Returns:
        list: Retrieved documents
    """
    # Simple keyword matching (for this demo)
    query_words = set(query.lower().split())

    scored_docs = []
    for doc in knowledge_base:
        doc_words = set(doc["content"].lower().split())
        score = len(query_words & doc_words)
        scored_docs.append((score, doc))

    # Sort by score and return top_k
    scored_docs.sort(reverse=True, key=lambda x: x[0])
    return [doc for score, doc in scored_docs[:top_k]]

def rag_query(question, knowledge_base, model="mistral-small-latest"):
    """
    Perform RAG: Retrieve relevant documents and generate answer

    Args:
        question (str): User's question
        knowledge_base (list): Knowledge base to search

    Returns:
        str: Generated answer with context
    """
    # Step 1: Retrieve relevant documents
    retrieved_docs = retrieval_demo(question, knowledge_base)

    # Step 2: Construct context from retrieved documents
    context = "\n\n".join([doc["content"] for doc in retrieved_docs])

    # Step 3: Create prompt with context
    rag_prompt = f"""Use the following context to answer the question. If the answer cannot be found in the context, say so.
                Context:
                {context}

                Question: {question}

                Answer:"""
                # This "rag_prompt" is for demo usage, please design your prompt.

    return basic_prompt(rag_prompt, model), retrieved_docs


# Example usage
question = "When was the Eiffel Tower built and why?"
print(f"Question: {question}\n")

answer, retrieved = rag_query(question, KNOWLEDGE_BASE)
print(f"Retrieved Documents:")
for doc in retrieved:
    print(f"- Doc {doc['id']}: {doc['content'][:50]}...")

print(f"\nAnswer:\n{answer}\n")

# Try another question
question2 = "What is machine learning?"
print(f"Question: {question2}\n")

answer2, retrieved2 = rag_query(question2, KNOWLEDGE_BASE)
print(f"Retrieved Documents:")
for doc in retrieved2:
    print(f"- Doc {doc['id']}: {doc['content'][:100]}...")

print(f"\nAnswer:\n{answer2}\n")

